# SSM-Causal: Run All Experiments on GPU

**Paper:** Causal Discovery via State Space Model Internals 
**Target:** NeurIPS 2026 
**Authors:** Ankit Hemant Lade, Aman Chadha

This notebook runs ALL 7 experiments on a free Colab T4 GPU (~30-45 min total).

**Instructions:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells (Ctrl+F9)
3. Results download automatically at the end

## 0. Setup: Verify GPU & Install Dependencies

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
%%bash
pip install -q statsmodels tigramite scikit-learn matplotlib seaborn networkx pandas scipy

## 1. Upload Project Code

Run the cell below — it creates the entire project structure inline.

In [ ]:
import os

# Create project structure
for d in ['ssm_causal', 'baselines', 'data', 'experiments', 'results',
          'results/core_validation', 'results/exp1_synthetic',
          'results/exp2_nonstationary', 'results/exp3_climate',
          'results/exp4_fmri', 'results/exp5_efficiency', 'results/ablation',
          'paper']:
    os.makedirs(d, exist_ok=True)

print("Project structure created.")
print("Now upload the project files...")

In [ ]:
# Upload from Google Drive or direct upload
# Option A: Mount Google Drive (if you uploaded the project there)
from google.colab import drive
drive.mount('/content/drive')

# Copy project from Drive
import shutil
DRIVE_PATH = '/content/drive/MyDrive/ssm_causal'  # Adjust this path
if os.path.exists(DRIVE_PATH):
    # Copy all .py files
    for root, dirs, files in os.walk(DRIVE_PATH):
        for f in files:
            if f.endswith('.py') or f.endswith('.json'):
                src = os.path.join(root, f)
                rel = os.path.relpath(src, DRIVE_PATH)
                dst = os.path.join('/content', rel)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                shutil.copy2(src, dst)
    print(f"Copied project from {DRIVE_PATH}")
else:
    print(f"Drive path {DRIVE_PATH} not found.")
    print("Will create all files inline below.")

## 2. Create All Source Files Inline

If Google Drive upload didn't work, this cell creates everything from scratch.

In [ ]:
# Check if files already exist (from Drive)
if os.path.exists('/content/ssm_causal/mamba_model.py'):
    print("Files already present from Drive. Skipping inline creation.")
else:
    print("Creating files inline — please upload your project to Google Drive")
    print("Path: My Drive/ssm_causal/")
    print("")
    print("Required files:")
    print("  ssm_causal/mamba_model.py")
    print("  ssm_causal/extractor.py")
    print("  ssm_causal/score.py")
    print("  ssm_causal/threshold.py")
    print("  ssm_causal/utils.py")
    print("  ssm_causal/__init__.py")
    print("  baselines/granger.py")
    print("  baselines/pcmci_wrapper.py")
    print("  baselines/neural_granger.py")
    print("  baselines/__init__.py")
    print("  data/synthetic.py")
    print("  data/climate.py")
    print("  data/fmri.py")
    print("  data/__init__.py")
    print("  experiments/exp0_core_validation.py")
    print("  experiments/exp1_synthetic.py")
    print("  experiments/exp2_nonstationary.py")
    print("  experiments/exp3_climate.py")
    print("  experiments/exp4_fmri.py")
    print("  experiments/exp5_efficiency.py")
    print("  experiments/exp6_ablation.py")
    print("  experiments/__init__.py")

## 3. Run All Experiments

Each experiment runs sequentially. Total time: ~30-45 min on T4 GPU.

In [ ]:
import time
os.chdir('/content')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on: {DEVICE}")
total_start = time.time()

### Exp 0: Core Validation (Day 3 Test)

In [ ]:
%%time
!python experiments/exp0_core_validation.py

### Exp 1: Full Synthetic Benchmark (Table 1)

In [ ]:
%%time
!python experiments/exp1_synthetic.py

### Exp 2: Regime-Switching (Table 2 + Figure 1)

In [ ]:
%%time
!python experiments/exp2_nonstationary.py

### Exp 3: Climate Teleconnections (Figure 2)

In [ ]:
%%time
!python experiments/exp3_climate.py

### Exp 4: fMRI Brain Connectivity (Figure 3)

In [ ]:
%%time
!python experiments/exp4_fmri.py

### Exp 5: Efficiency & Scalability (Table 4)

In [ ]:
%%time
!python experiments/exp5_efficiency.py

### Exp 6: Ablation Study (Table 5)

In [ ]:
%%time
!python experiments/exp6_ablation.py

## 4. Summary & Download Results

In [ ]:
total_time = time.time() - total_start
print(f"\n{'='*60}")
print(f"ALL EXPERIMENTS COMPLETE")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Device: {DEVICE}")
print(f"{'='*60}")

# List all result files
import glob
print("\nResult files:")
for f in sorted(glob.glob('results/**/*', recursive=True)):
    if os.path.isfile(f):
        size = os.path.getsize(f)
        print(f"  {f} ({size:,} bytes)")

In [ ]:
# Zip all results
!zip -r /content/ssm_causal_results.zip results/

# Copy to Google Drive
import shutil
drive_results = '/content/drive/MyDrive/ssm_causal_results'
os.makedirs(drive_results, exist_ok=True)
shutil.copy('/content/ssm_causal_results.zip', drive_results)

# Also copy individual result folders
for d in glob.glob('results/*'):
    if os.path.isdir(d):
        dst = os.path.join(drive_results, os.path.basename(d))
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(d, dst)

print(f"\nResults saved to Google Drive: {drive_results}")
print("You can also download the zip:")

In [ ]:
# Download zip to your local machine
from google.colab import files
files.download('/content/ssm_causal_results.zip')

## Done!

Results are saved to:
- Google Drive: `My Drive/ssm_causal_results/`
- Downloaded zip: `ssm_causal_results.zip`

Copy the results back to your local project:
```bash
unzip ssm_causal_results.zip -d /path/to/ssm_causal/
python paper/generate_tables.py  # Auto-generate LaTeX tables
```